# EDA — Rendimiento estudiantil
Análisis exploratorio de la base de estudiantes ya limpia.

**Dataset:** cada fila es un estudiante, con columnas como `gender`, `parental_education`,
`study_time_hours`, `attendance_percent`, `previous_grade`, `final_exam_score`, `final_grade`, etc.

## Paso 0 · Cargar los datos
Subo el Excel con la base limpia y lo leo con pandas.

In [ ]:
import pandas as pd

# Si trabajas en Colab, descomenta para subir el archivo:
# from google.colab import files
# subido = files.upload()

df = pd.read_excel('student_performance_limpio.xlsx', sheet_name='student_performance_original')
df.head()

## Paso 1 · Entendimiento inicial
❯ ¿Cuántas filas y columnas hay, y qué tipo de dato es cada una?

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df.describe()

## Paso 2 · Calidad de datos
❯ ¿Hay datos faltantes, duplicados o valores imposibles?

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
# columnas numéricas que voy a analizar
num = ['study_time_hours', 'attendance_percent', 'sleep_hours', 'previous_grade', 'final_exam_score']
df[num].describe()

In [ ]:
# En el describe se ven valores imposibles (nota 999, asistencia 150, etc.)
# que quedaron de la limpieza. Los quito para que no dañen el análisis.
df = df[(df['final_exam_score'] <= 100) &
        (df['attendance_percent'] <= 100) &
        (df['study_time_hours'] <= 12) &
        (df['previous_grade'] <= 100) &
        (df['sleep_hours'] <= 12)]
df.shape

## Paso 3 · Análisis univariado
❯ ¿Cómo se distribuye la nota del examen final? ¿Hay notas atípicas?

In [ ]:
df['final_exam_score'].describe()

In [ ]:
df['final_exam_score'].plot(kind='hist', bins=20, title='Distribución de la nota final')

In [ ]:
df.boxplot(column='final_exam_score')

In [ ]:
df['final_grade'].value_counts().sort_index().plot(kind='bar', title='Estudiantes por nota final')

## Paso 4 · Análisis bivariado
❯ ¿Estudiar más se relaciona con mejor nota? ¿Influye la educación de los padres?

In [ ]:
df.plot.scatter(x='study_time_hours', y='final_exam_score', title='Horas de estudio vs nota final')

In [ ]:
df.groupby('parental_education')['final_exam_score'].mean().plot(kind='bar', title='Nota promedio según educación de los padres')

In [ ]:
df.groupby('gender')['final_exam_score'].mean()

## Paso 5 · Correlación
❯ ¿Qué variable se relaciona más con la nota final?

In [ ]:
df[num].corr()['final_exam_score'].sort_values(ascending=False)

In [ ]:
df.plot.scatter(x='previous_grade', y='final_exam_score', title='Nota previa vs nota final')

## Paso 6 · Conclusiones
❯ ¿Qué le dirías a alguien que quiere mejorar las notas con estos datos?

**Hallazgo 1:** Las horas de estudio son la variable que más se relaciona con la nota final
(correlación ≈ 0.63). A más horas de estudio, mejor nota. La nota previa también ayuda (≈ 0.42).

**Hallazgo 2:** La educación de los padres y el género casi no cambian la nota promedio
(todo queda alrededor de 73–74), así que no parecen factores importantes aquí.

**Calidad de datos:** Encontré una nota de 999 y otros valores imposibles (asistencia 150,
sueño 25 h) que quedaron de la limpieza. Los quité antes de analizar, pero habría que
corregirlos en el origen.

**Hipótesis a validar:** Si un estudiante aumenta sus horas de estudio, debería subir su nota.
Habría que comprobarlo con un modelo, no solo con la correlación.